# Wrap into a Vertex AI Pipeline (Kubeflow Pipelines - KFP)

### Libraries

In [ ]:
from typing import List

from google.cloud import aiplatform
from kfp import dsl, compiler
from kfp.dsl import component, Input, Output, Artifact, Model, Metrics
from google_cloud_pipeline_components.v1.vertex_notification_email import VertexNotificationEmailOp

### Project setup

In [ ]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
BUCKET = PROJECT_ID
REGION = "us-central1"
PIPELINE_ROOT = f"gs://{BUCKET}/pipeline_root"

aiplatform.init(project=PROJECT_ID, location=REGION)
email_ids = ["sandeep.raju.anantha@gmail.com"]
dagshub_url= "https://dagshub.com/Sandeepraju-42/Forecast_MLFlow_Fashion_Dataset.mlflow"

### COMPONENT 1: Refresh & Feature Preparation

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "google-cloud-storage", "google-cloud-bigquery"]
)
def prepare_data_op(gcs_output_path: str):
    import pandas as pd
    import re

    # Fetch fresh data from BigQuery/GCS and apply cleaning rules
    # df = fetch_fresh_data()
    # df.columns = [re.sub(r'[\W]+', '_', col).strip('_') for col in df.columns]
    # df.to_csv(gcs_output_path, index=False)
    print(f"Data prepared and saved to {gcs_output_path}")

### COMPONENT 2: Train Custom Candidate & Log to MLflow

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=[
        "setuptools<81",  # xgboost==1.6.2's compat.py needs pkg_resources; recent setuptools drops it
        "pandas", "numpy", "xgboost==1.6.2", "mlflow", "google-cloud-aiplatform", "google-cloud-storage",
    ]
)
def train_xgboost_candidate_op(
    gcs_data_path: str,
    mlflow_tracking_uri: str,
    metrics_output: Output[Metrics],
    model_artifact: Output[Model],
) -> float:
    import os

    import numpy as np
    import pandas as pd
    import xgboost as xgb
    from google.cloud import storage

    # MLflow logging
    mlflow_enabled = bool(mlflow_tracking_uri)
    if mlflow_enabled:
        try:
            import mlflow

            mlflow.set_tracking_uri(mlflow_tracking_uri)
            mlflow.set_experiment("vertex-pipeline-runs")
        except Exception as e:
            print(f"MLflow unavailable ({e!r}) -- continuing without it.")
            mlflow_enabled = False

    # --- Load data -----------------------------------------------------
    # google-cloud-storage instead of pandas
    assert gcs_data_path.startswith("gs://"), f"expected a gs:// path, got {gcs_data_path!r}"
    bucket_name, blob_path = gcs_data_path[len("gs://"):].split("/", 1)
    local_csv = "/tmp/training_data.csv"
    storage.Client().bucket(bucket_name).blob(blob_path).download_to_filename(local_csv)
    df = pd.read_csv(local_csv, parse_dates=["date"])
    print(f"Loaded {len(df):,} rows from {gcs_data_path}")

    # --- Columns ---------------------------------------------------------
    # Same TARGET_COLUMN/TIME_COLUMN and numeric feature
    TARGET_COLUMN = "sales_qty"
    TIME_COLUMN = "date"
    FEATURE_COLUMNS = [c for c in [
        "price", "discount_pct", "competitor_price_index", "week_of_year", "month",
        "quarter", "days_since_launch", "temperature", "precipitation", "marketing_spend",
        "lead_time_days", "store_count", "cannibalization_index", "substitute_price_ratio",
        "woy_sin", "woy_cos", "month_sin", "month_cos", "price_change_pct",
        "relative_price_vs_category", "weeks_since_last_promo", "sales_qty_lag_8",
        "sales_qty_lag_9", "sales_qty_lag_10", "sales_qty_lag_12", "sales_qty_lag_52",
        "sales_qty_rollmean_4", "sales_qty_rollstd_4", "sales_qty_rollmean_12",
        "sales_qty_rollstd_12", "review_count_lag8", "avg_rating_lag8",
        "wishlist_adds_lag8", "social_trend_score_lag8", "inventory_on_hand_lag8",
    ] if c in df.columns]
    missing = [c for c in ["price", "discount_pct", "week_of_year"] if c not in df.columns]
    if missing:
        raise KeyError(f"Expected columns not found in {gcs_data_path}: {missing}")

    df = df.dropna(subset=[TARGET_COLUMN])
    df[FEATURE_COLUMNS] = df[FEATURE_COLUMNS].fillna(0)

    # --- Time-based holdout split ---------------------------------------
    # A single global date cutoff, not per-SKU -- leakage-free (validation
    # rows are all strictly after training rows)
    cutoff = df[TIME_COLUMN].quantile(0.9)
    train_df = df[df[TIME_COLUMN] <= cutoff]
    val_df = df[df[TIME_COLUMN] > cutoff]
    print(f"Train: {len(train_df):,} rows (<= {cutoff.date()}), "
          f"Val: {len(val_df):,} rows (> {cutoff.date()})")

    X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
    X_val, y_val = val_df[FEATURE_COLUMNS], val_df[TARGET_COLUMN]

    bst = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    bst.fit(X_train, y_train)

    preds = np.clip(bst.predict(X_val), a_min=0, a_max=None)
    actual = y_val.to_numpy()
    val_wape = float(np.abs(actual - preds).sum() / max(actual.sum(), 1e-9))
    print(f"val_wape: {val_wape:.4f}")

    # model_artifact.uri is pre-assigned by KFP to a GCS directory; write the
    # actual model file to model_artifact.path
    os.makedirs(model_artifact.path, exist_ok=True)
    bst.get_booster().save_model(os.path.join(model_artifact.path, "model.bst"))
    print(f"Saved model.bst to {model_artifact.path}")

    if mlflow_enabled:
        import mlflow

        try:
            with mlflow.start_run(run_name="pipeline_candidate_xgb"):
                mlflow.log_metric("val_wape", val_wape)
                mlflow.log_params({"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05})
        except Exception as e:
            print(f"MLflow logging failed ({e!r}) -- training result is unaffected.")
    else:
        print("mlflow_tracking_uri not set/reachable -- skipping MLflow, training proceeds anyway.")

    metrics_output.log_metric("val_wape", val_wape)
    return val_wape

### COMPONENT 3: Champion vs Candidate Evaluator

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"]
)
def evaluate_and_compare_op(
    candidate_wape: float,
    model_display_name: str,
    project: str,
    location: str,
) -> bool:
    from google.cloud import aiplatform

    aiplatform.init(project=project, location=location)

    # Fetch current champion model metrics from Model Registry.
    models = aiplatform.Model.list(
        filter=f'display_name="{model_display_name}"',
        order_by="create_time desc",
    )

    if not models:
        print("No champion model found. Candidate defaults to WINNER.")
        return True

    champion_model = models[0]

    # Real champion WAPE, read back from the label deploy_model_op sets
    wape_bps = (champion_model.labels or {}).get("val_wape_bps")
    if wape_bps is not None:
        champion_wape = int(wape_bps) / 10000
    else:
        champion_wape = 0.100
        print("Champion has no stored val_wape_bps label (deployed before this "
              "fix) -- falling back to the 0.100 default threshold.")

    print(f"Candidate WAPE: {candidate_wape:.4f} | Champion WAPE: {champion_wape:.4f}")

    # Returns True if candidate outperforms champion
    return candidate_wape < champion_wape

### COMPONENT 4: Register & Deploy Model

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"]
)
def deploy_model_op(
    model_artifact: Input[Model],
    model_display_name: str,
    endpoint_name: str,
    project: str,
    location: str,
    candidate_wape: float,
):
    import time

    from google.api_core.exceptions import InternalServerError, ServiceUnavailable
    from google.cloud import aiplatform

    aiplatform.init(project=project, location=location)

    # Register new version in Vertex AI Model Registry. val_wape_bps stores
    # this run's real WAPE as basis points (round(wape * 10000))
    registered_model = aiplatform.Model.upload(
        display_name=model_display_name,
        artifact_uri=model_artifact.uri,
        labels={"val_wape_bps": str(round(candidate_wape * 10000))},
        serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-6:latest"
    )

    # Get or create Endpoint
    endpoints = aiplatform.Endpoint.list(filter=f'display_name="{endpoint_name}"')
    if endpoints:
        endpoint = endpoints[0]
    else:
        endpoint = aiplatform.Endpoint.create(display_name=endpoint_name)

    # Deploy winner model to endpoint with 100% traffic. 
    max_attempts = 3
    for attempt in range(1, max_attempts + 1):
        try:
            registered_model.deploy(
                endpoint=endpoint,
                traffic_percentage=100,
                sync=True
            )
            break
        except (InternalServerError, ServiceUnavailable) as e:
            if attempt == max_attempts:
                raise

            # Distributed-systems hazard
            refreshed_endpoint = aiplatform.Endpoint(endpoint.resource_name)
            already_deployed = any(
                dm.model == registered_model.resource_name
                for dm in refreshed_endpoint.gca_resource.deployed_models
            )
            if already_deployed:
                print(f"Deploy attempt {attempt} reported an error ({e!r}) but the "
                      "model is already on the endpoint -- treating it as deployed.")
                endpoint = refreshed_endpoint
                break

            backoff_seconds = 30 * (2 ** (attempt - 1))  # 30s, 60s, 120s
            print(f"Deploy attempt {attempt}/{max_attempts} hit a transient "
                  f"error ({e!r}) -- retrying in {backoff_seconds}s.")
            time.sleep(backoff_seconds)

    print(f"Model deployed to endpoint: {endpoint.resource_name}")

### COMPONENT 5: Notify on Promotion

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-pubsub"]
)
def notify_promotion_op(
    project: str,
    topic_id: str,
    model_display_name: str,
    endpoint_name: str,
    candidate_wape: float,
):
    import json
    from google.cloud import pubsub_v1

    publisher = pubsub_v1.PublisherClient()
    topic_path = publisher.topic_path(project, topic_id)
    message = {
        "event": "model_promoted",
        "model_display_name": model_display_name,
        "endpoint_name": endpoint_name,
        "candidate_wape": candidate_wape,
    }
    future = publisher.publish(topic_path, json.dumps(message).encode("utf-8"))
    future.result()  # block until the publish actually lands, so failures surface here, not silently
    print(f"Published promotion notification to {topic_path}")

### COMPONENT 6: Batch Forecast to BigQuery

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=[
        "setuptools<81",  # xgboost==1.6.2's compat.py needs pkg_resources; recent setuptools drops it
        "pandas", "pyarrow", "xgboost==1.6.2",
        "google-cloud-aiplatform", "google-cloud-bigquery", "google-cloud-storage",
    ]
)
def batch_forecast_to_bq_op(
    project: str,
    location: str,
    model_display_name: str,
    gcs_data_path: str,
    bq_dataset: str,
    forecast_horizon: int = 8,
):
    import datetime

    import numpy as np
    import pandas as pd
    import xgboost as xgb
    from google.cloud import aiplatform, bigquery, storage

    aiplatform.init(project=project, location=location)

    # Same fixed lookup as evaluate_and_compare_op
    champion_models = aiplatform.Model.list(
        filter=f'display_name="{model_display_name}"',
        order_by="create_time desc",
    )
    if not champion_models:
        print(f"No registered model named {model_display_name} yet -- nothing to forecast.")
        return
    champion = champion_models[0]

    if not champion.uri:
        print(f"Champion model {champion.resource_name} has no artifact_uri -- nothing to forecast.")
        return

    # Download the champion's real model.bst and load it
    champ_bucket, champ_prefix = champion.uri[len("gs://"):].split("/", 1)
    local_model_path = "/tmp/champion_model.bst"
    storage.Client().bucket(champ_bucket).blob(f"{champ_prefix.rstrip('/')}/model.bst").download_to_filename(local_model_path)
    bst = xgb.Booster()
    bst.load_model(local_model_path)

    # Download the same feature file training reads from.
    assert gcs_data_path.startswith("gs://"), f"expected a gs:// path, got {gcs_data_path!r}"
    data_bucket, data_blob = gcs_data_path[len("gs://"):].split("/", 1)
    local_csv = "/tmp/forecast_input.csv"
    storage.Client().bucket(data_bucket).blob(data_blob).download_to_filename(local_csv)
    features_df = pd.read_csv(local_csv, parse_dates=["date"])

    TIME_COLUMN, SERIES_ID_COLUMN, TARGET_COLUMN = "date", "sku_id", "sales_qty"
    # Same numeric feature set train_xgboost_candidate_op trains on
    FEATURE_COLUMNS = [c for c in [
        "price", "discount_pct", "competitor_price_index", "week_of_year", "month",
        "quarter", "days_since_launch", "temperature", "precipitation", "marketing_spend",
        "lead_time_days", "store_count", "cannibalization_index", "substitute_price_ratio",
        "woy_sin", "woy_cos", "month_sin", "month_cos", "price_change_pct",
        "relative_price_vs_category", "weeks_since_last_promo", "sales_qty_lag_8",
        "sales_qty_lag_9", "sales_qty_lag_10", "sales_qty_lag_12", "sales_qty_lag_52",
        "sales_qty_rollmean_4", "sales_qty_rollstd_4", "sales_qty_rollmean_12",
        "sales_qty_rollstd_12", "review_count_lag8", "avg_rating_lag8",
        "wishlist_adds_lag8", "social_trend_score_lag8", "inventory_on_hand_lag8",
    ] if c in features_df.columns]

    # feature -> (source column to lag, lag periods). 
    LAG_SPECS = {
        "sales_qty_lag_8": (TARGET_COLUMN, 8),
        "sales_qty_lag_9": (TARGET_COLUMN, 9),
        "sales_qty_lag_10": (TARGET_COLUMN, 10),
        "sales_qty_lag_12": (TARGET_COLUMN, 12),
        "sales_qty_lag_52": (TARGET_COLUMN, 52),
        "review_count_lag8": ("review_count", 8),
        "avg_rating_lag8": ("avg_rating", 8),
        "wishlist_adds_lag8": ("wishlist_adds", 8),
        "social_trend_score_lag8": ("social_trend_score", 8),
        "inventory_on_hand_lag8": ("inventory_on_hand", 8),
    }
    # feature -> (window size, stat)
    ROLL_SPECS = {
        "sales_qty_rollmean_4": (4, "mean"),
        "sales_qty_rollstd_4": (4, "std"),
        "sales_qty_rollmean_12": (12, "mean"),
        "sales_qty_rollstd_12": (12, "std"),
    }

    # Cadence inferred from the data itself (this project's data is
    # weekly, but this avoids silently assuming that if it ever changes).
    all_dates = np.sort(features_df[TIME_COLUMN].unique())
    if len(all_dates) >= 2:
        days_per_period = int(np.median(np.diff(all_dates).astype("timedelta64[D]").astype(int)))
    else:
        days_per_period = 7
    period = pd.Timedelta(days=days_per_period)

    generated_at = datetime.datetime.utcnow()
    output_rows = []

    for sku_id, group in features_df.groupby(SERIES_ID_COLUMN):
        group = group.sort_values(TIME_COLUMN).reset_index(drop=True)
        last_row = group.iloc[-1]
        last_date = last_row[TIME_COLUMN]

        # Running history per source column 
        history = {TARGET_COLUMN: list(group[TARGET_COLUMN].values)}
        for _, (base_col, _lag) in LAG_SPECS.items():
            if base_col not in history:
                history[base_col] = list(group[base_col].values) if base_col in group.columns else None

        for h in range(1, forecast_horizon + 1):
            forecast_date = last_date + h * period
            row = {}

            if "week_of_year" in FEATURE_COLUMNS:
                row["week_of_year"] = forecast_date.isocalendar()[1]
                row["month"] = forecast_date.month
                row["quarter"] = forecast_date.quarter
                row["woy_sin"] = np.sin(2 * np.pi * row["week_of_year"] / 52)
                row["woy_cos"] = np.cos(2 * np.pi * row["week_of_year"] / 52)
                row["month_sin"] = np.sin(2 * np.pi * row["month"] / 12)
                row["month_cos"] = np.cos(2 * np.pi * row["month"] / 12)
            if "days_since_launch" in FEATURE_COLUMNS:
                row["days_since_launch"] = last_row["days_since_launch"] + h * days_per_period

            for lag_col, (base_col, k) in LAG_SPECS.items():
                if lag_col not in FEATURE_COLUMNS:
                    continue
                source = history.get(base_col)
                if source is not None:
                    lookback_idx = len(source) - k
                    row[lag_col] = source[lookback_idx] if lookback_idx >= 0 else 0.0
                else:
                    # Raw column not in this feature file -- fall back to
                    # the last known lagged value rather than guessing.
                    row[lag_col] = last_row[lag_col]

            target_history = history[TARGET_COLUMN]

            roll_end_idx = len(target_history) - forecast_horizon
            for roll_col, (w, stat) in ROLL_SPECS.items():
                if roll_col not in FEATURE_COLUMNS:
                    continue
                if roll_end_idx < 0:
                    window_vals = []
                else:
                    roll_start_idx = max(0, roll_end_idx - w + 1)
                    window_vals = target_history[roll_start_idx:roll_end_idx + 1]
                if stat == "mean":
                    row[roll_col] = float(np.mean(window_vals)) if window_vals else 0.0
                else:
                    row[roll_col] = float(np.std(window_vals, ddof=1)) if len(window_vals) > 1 else 0.0


            for c in FEATURE_COLUMNS:
                if c not in row:
                    row[c] = last_row[c]

            X = pd.DataFrame([row])[FEATURE_COLUMNS].fillna(0)
            pred = float(bst.predict(xgb.DMatrix(X))[0])

            output_rows.append({
                "sku_id": sku_id,
                "date": forecast_date,
                "predicted_sales_qty": pred,
                "model_display_name": model_display_name,
                "model_resource_name": champion.resource_name,
                "generated_at": generated_at,
            })

            history[TARGET_COLUMN] = target_history + [pred]

    forecast_df = pd.DataFrame(output_rows)

    # ---- Actual vs. predicted + the forecast_horizon rows above, at SKU
    # level, written to GCS as one CSV
    historical_preds = bst.predict(xgb.DMatrix(features_df[FEATURE_COLUMNS].fillna(0)))
    actual_vs_predicted_df = pd.concat([
        pd.DataFrame({
            "sku_id": features_df[SERIES_ID_COLUMN].to_numpy(),
            "date": features_df[TIME_COLUMN].to_numpy(),
            "actual_sales_qty": features_df[TARGET_COLUMN].to_numpy(),
            "predicted_sales_qty": historical_preds,
            "is_forecast": False,
        }),
        pd.DataFrame([
            {
                "sku_id": r["sku_id"],
                "date": r["date"],
                "actual_sales_qty": np.nan,
                "predicted_sales_qty": r["predicted_sales_qty"],
                "is_forecast": True,
            }
            for r in output_rows
        ]),
    ], ignore_index=True).sort_values([SERIES_ID_COLUMN, TIME_COLUMN])

    # Same bucket gcs_data_path
    output_blob_path = "data/output/actual_vs_predicted.csv"
    storage.Client().bucket(data_bucket).blob(output_blob_path).upload_from_string(
        actual_vs_predicted_df.to_csv(index=False), content_type="text/csv"
    )
    print(f"Wrote {len(actual_vs_predicted_df)} actual-vs-predicted rows to "
          f"gs://{data_bucket}/{output_blob_path}")

    bq_client = bigquery.Client(project=project)

    # latest_forecasts: Power BI dataset.
    bq_client.load_table_from_dataframe(
        forecast_df,
        f"{project}.{bq_dataset}.latest_forecasts",
        job_config=bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE),
    ).result()

    # forecast_history: accumulating log
    bq_client.load_table_from_dataframe(
        forecast_df,
        f"{project}.{bq_dataset}.forecast_history",
        job_config=bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_APPEND),
    ).result()

    print(f"Wrote {len(forecast_df)} forecast rows ({forecast_horizon} periods x "
          f"{features_df[SERIES_ID_COLUMN].nunique()} SKUs) to {bq_dataset}.latest_forecasts and .forecast_history")

# PIPELINE DEFINITION

In [ ]:
@dsl.pipeline(
    name="forecasting-champion-challenger-pipeline",
    description="Automated retraining, evaluation, MLflow logging, and conditional deployment."
)
def forecasting_pipeline(
    gcs_data_path: str,
    mlflow_tracking_uri: str,
    project: str,
    notification_emails: List[str],
    pubsub_topic_id: str = "pipeline-notifications",
    bq_dataset: str = "forecasting",
    location: str = "us-central1",
    model_display_name: str = "fashion_demand_forecasting",
    endpoint_name: str = "fashion_demand_endpoint",
):
    # VertexNotificationEmailOp
    notify_email_task = VertexNotificationEmailOp(recipients=notification_emails)
    with dsl.ExitHandler(notify_email_task, name="notify-on-completion"):

        # Step 1: Refresh & Clean Data
        prep_task = prepare_data_op(gcs_output_path=gcs_data_path)

        # Step 2: Train Candidate Model
        train_task = train_xgboost_candidate_op(
            gcs_data_path=gcs_data_path,
            mlflow_tracking_uri=mlflow_tracking_uri
        ).after(prep_task)

        # Step 3: Compare Candidate vs. Champion
        eval_task = evaluate_and_compare_op(
            candidate_wape=train_task.outputs["Output"],
            model_display_name=model_display_name,
            project=project,
            location=location,
        )

        # Step 4: Conditional Deployment (Only if Candidate beats Champion)
        with dsl.If(
            eval_task.output == True,
            name="candidate-is-better"
        ):
            deploy_task = deploy_model_op(
                model_artifact=train_task.outputs["model_artifact"],
                model_display_name=model_display_name,
                endpoint_name=endpoint_name,
                project=project,
                location=location,
                candidate_wape=train_task.outputs["Output"],
            )

            # Step 5: Tell whatever's downstream that a new model just went live.
            notify_promotion_task = notify_promotion_op(
                project=project,
                topic_id=pubsub_topic_id,
                model_display_name=model_display_name,
                endpoint_name=endpoint_name,
                candidate_wape=train_task.outputs["Output"],
            ).after(deploy_task)

            # Step 6a: candidate won
            batch_forecast_task_won = batch_forecast_to_bq_op(
                project=project,
                location=location,
                model_display_name=model_display_name,
                gcs_data_path=gcs_data_path,
                bq_dataset=bq_dataset,
            ).after(deploy_task)

        with dsl.Else(name="candidate-did-not-win"):

            # Step 6b: candidate lost
            batch_forecast_task_lost = batch_forecast_to_bq_op(
                project=project,
                location=location,
                model_display_name=model_display_name,
                gcs_data_path=gcs_data_path,
                bq_dataset=bq_dataset,
            )

# COMPILE AND RUN ON VERTEX AI PIPELINES

In [ ]:
compiler.Compiler().compile(
    pipeline_func=forecasting_pipeline,
    package_path="forecasting_pipeline.yaml"
)

# One-time, idempotent setup
from google.api_core.exceptions import AlreadyExists
from google.cloud import bigquery, pubsub_v1

PUBSUB_TOPIC_ID = "pipeline-notifications"
_publisher = pubsub_v1.PublisherClient()
_topic_path = _publisher.topic_path(PROJECT_ID, PUBSUB_TOPIC_ID)
try:
    _publisher.create_topic(name=_topic_path)
    print(f"Created topic {_topic_path}")
except AlreadyExists:
    print(f"Topic {_topic_path} already exists -- nothing to do")

BQ_DATASET = "forecasting"
_bq_client = bigquery.Client(project=PROJECT_ID)
_bq_client.create_dataset(f"{PROJECT_ID}.{BQ_DATASET}", exists_ok=True)
print(f"Dataset {PROJECT_ID}.{BQ_DATASET} ready "
      "(tables inside it -- latest_forecasts, forecast_history -- are "
      "created automatically on first write by load_table_from_dataframe)")

# Submit pipeline job to Vertex AI
GCS_DATA_PATH = f"gs://{BUCKET}/data/input/synthetic_fashion_demand_features.csv"
MLFLOW_TRACKING_URI = dagshub_url  # runs fine without it
NOTIFICATION_EMAILS = email_ids 
MODEL_DISPLAY_NAME = "fashion_demand_forecasting"  # matches the pipeline's own default
ENDPOINT_NAME = "fashion_demand_endpoint"           # matches the pipeline's own default

PIPELINE_PARAMETER_VALUES = {
    "gcs_data_path": GCS_DATA_PATH,
    "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
    "project": PROJECT_ID,
    "location": REGION,
    "notification_emails": NOTIFICATION_EMAILS,
    "pubsub_topic_id": PUBSUB_TOPIC_ID,
    "bq_dataset": BQ_DATASET,
    "model_display_name": MODEL_DISPLAY_NAME,
    "endpoint_name": ENDPOINT_NAME,
}

pipeline_job = aiplatform.PipelineJob(
    display_name="demand_forecasting_production_run",
    template_path="forecasting_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values=PIPELINE_PARAMETER_VALUES,
)

pipeline_job.run(sync=False)

# Step 7 -- Put it on a schedule using aiplatform.PipelineJobSchedule

In [ ]:
CREATE_SCHEDULE = False  # <- flip to True once you want this running unattended

# Reuses the same PipelineJob object built above
pipeline_job_for_schedule = aiplatform.PipelineJob(
    display_name="demand_forecasting_scheduled_run",
    template_path="forecasting_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values=PIPELINE_PARAMETER_VALUES,
)

if CREATE_SCHEDULE:
    pipeline_job_schedule = aiplatform.PipelineJobSchedule(
        pipeline_job=pipeline_job_for_schedule,
        display_name="forecasting-pipeline-weekly",
    )
    pipeline_job_schedule.create(
        # Standing in for "new data arrived" 
        cron="TZ=Europe/Stockholm 0 6 * * MON",
        max_concurrent_run_count=1,

        # No dedicated service_account set here 
    )
    print(f"Created schedule: {pipeline_job_schedule.resource_name}")
else:
    print("CREATE_SCHEDULE is False -- nothing scheduled, nothing billed periodically.")

### Managing an existing schedule
pausing costs nothing further; 
deleting removes the resource entirely.

In [ ]:
schedules = aiplatform.PipelineJobSchedule.list(
    filter=f'display_name="forecasting-pipeline-weekly"'
)
for sched in schedules:
    print(sched.resource_name, sched.state)

# sched.pause()    # stop future runs, keep the resource and its history
# sched.resume()   # resume a paused schedule
# sched.delete()   # remove it entirely